In [0]:
USE CATALOG de_journey;
USE SCHEMA practice;

In [0]:
---- ORDER OF EXECUTION ----

-- #6+7   SELECT DISTINCT department_id                                 
-- #1     FROM employees                                                
-- #2     JOIN orders ON customers.customer_id = orders.customer_id     
-- #3     WHERE salary > 3000                                          
-- #4     GROUP BY department 
-- #5     HAVING AVG(salary) > 5000 
-- #8     ORDER BY department 
-- #9     LIMIT 10 OFFSET 5 
-- #10    OFFSET 5 ROWS FETCH NEXT 10 ROWS ONLY; 

# Beginner Level

### Q1. Find all customers and their order amounts

In [0]:

SELECT c.customer_id, name, sum(o.amount)
FROM customers c
JOIN orders o
ON c.customer_id = o.customer_id
GROUP BY c.customer_id, name

### Q2. Find customers who never placed an order (E-commerce)

In [0]:
SELECT c.customer_id FROM
customers c LEFT JOIN
orders o
ON c.customer_id = o.customer_id
WHERE o.order_id is NULL

In [0]:
-- SELECT c.customer_id FROM
-- customers c LEFT JOIN
-- orders o
-- ON c.customer_id = o.customer_id
-- WHERE o.order_id is NULL

SELECT c.customer_id FROM
customers c LEFT ANTI JOIN
orders o
ON c.customer_id = o.customer_id

# Intermediate Level


### Q3. Get customer name, product name, and order amount

Three-table JOIN for order details (FAANG)

In [0]:
SELECT c.customer_name, p.product_name, o.amount as total_revenue FROM
customers c
JOIN orders o
ON c.customer_id = o.customer_id
JOIN products p
ON o.product_id = p.product_id
-- GROUP BY c.customer_name, p.product_name
-- ORDER BY total_revenue DESC

### Q4. Find products never ordered (E-commerce)



In [0]:
-- SLOWEST

-- SELECT p.product_id
-- FROM products p
-- EXCEPT
-- SELECT o.product_id
-- FROM orders o


-- SELECT p.product_id, p.product_name
-- FROM products p
-- LEFT ANTI JOIN orders o
-- ON p.product_id = o.product_id


-- SELECT p.product_name
-- FROM products p
-- WHERE p.product_id NOT IN (
--   SELECT o.product_id
--   FROM orders o
-- )

-- Best if not indexed

SELECT p.product_name 
from products p 
LEFT JOIN orders o ON 
p.product_id = o.product_id
WHERE o.order_id IS NULL;

-- Best if indexed or not

SELECT p.product_name 
FROM products p
WHERE NOT EXISTS (
    SELECT 1 FROM orders o WHERE p.product_id = o.product_id
)

https://www.sqlshack.com/t-sql-commands-performance-comparison-not-vs-not-exists-vs-left-join-vs-except/ 

# Advanced Level


### Q5. Total revenue per customer category

In [0]:
SELECT c.category, sum(o.amount)
from customers c
join orders o
on c.customer_id = o.customer_id
group by c.category;


### Q6. Find employee-manager pairs (FAANG)

In [0]:
SELECT e1.name as Reportee, e2.name as Manager
FROM employees e1 JOIN
employees e2 ON e1.manager_id = e2.employee_id
ORDER BY e1.name

-- if we dont need null values for managers


In [0]:
SELECT e1.name as Reportee, e2.name as Manager
FROM employees e1 LEFT JOIN
employees e2 ON e1.manager_id = e2.employee_id
ORDER BY e1.name

-- to show complete hierarchy including null values for managers


### Q7. Match transactions based on amount AND date (Finance)

In [0]:
SELECT * FROM transactions LIMIT 10

In [0]:
-- SELF JOIN

SELECT t1.transaction_id as T1, t2.transaction_id as T2, t1.amount, t2.amount
FROM transactions t1 JOIN transactions t2
ON t1.amount = t2.amount
and t1.transaction_date = t2.transaction_date
and t1.transaction_id < t2.transaction_id

-- t1.transaction_id = t2.transaction_id (WRONG it will just show the same id for t1 and t2 and again the same amount for t1 and t2)
-- t1.transaction_id <> t2.transaction_id (WRONG it will just show the ids but the reverse pairings will be present eg: 1,2 and 2,1)
-- t1.transaction_id < t2.transaction_id (CORRECT it will just show the ids but the reverse pairings will not be present eg: 1,2 but not 2,1)

### Q8. Full OUTER JOIN for reconciliation/mismatches in amount between 2 systems (Finance)


In [0]:
SELECT s1.id, s1.amount, s2.id, s2.amount
FROM system1 s1 FULL OUTER JOIN system2 s2
ON s1.id = s2.id;
-- WHERE s1.amount != s2.amount




In [0]:
SELECT s1.id, s1.amount, s2.id, s2.amount
FROM system1 s1 FULL OUTER JOIN system2 s2
ON s1.id = s2.id
WHERE s1.amount != s2.amount;

In [0]:
SELECT s1.id, s1.amount, s2.id, s2.amount
FROM system1 s1 FULL OUTER JOIN system2 s2
ON s1.id = s2.id
and s1.amount != s2.amount;

-- moving the amount condition to the where clause will not work because it will break the logic of the join

In [0]:
-- INNER JOIN 

SELECT s1.id, s1.amount, s2.id, s2.amount
FROM system1 s1 JOIN system2 s2
ON s1.id = s2.id
and s1.amount != s2.amount;

In [0]:
SELECT s1.id, s1.amount, s2.id, s2.amount
FROM system1 s1 FULL OUTER JOIN system2 s2
ON s1.id = s2.id
WHERE s1.amount != s2.amount OR s1.amount IS NULL OR s2.amount IS NULL;

In [0]:
SELECT 
  COALESCE(s1.id, s2.id) AS id,
  s1.amount AS system1_amount,
  s2.amount AS system2_amount
FROM system1 s1
FULL OUTER JOIN system2 s2 ON s1.id = s2.id
WHERE s1.amount != s2.amount OR s1.id IS NULL OR s2.id IS NULL;

### Q9. Generate all possible product-color combinations

In [0]:
-- Task: Generate all possible product-color combinations

SELECT p.product_name, c.color
FROM products p
CROSS JOIN colors c;

### Q10. Customer lifetime value with order metrics

In [0]:
SELECT c.customer_id, c.customer_name
, count(o.order_id) as total_orders
, sum(o.amount) as total_spent
, avg(o.amount) as avg_spent
FROM customers c
LEFT JOIN orders o
ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.customer_name
ORDER BY c.customer_name

In [0]:
-- INNER JOIN (to eliminate nulls)

SELECT c.customer_id, c.customer_name
, count(o.order_id) as total_orders
, sum(o.amount) as total_spent
, avg(o.amount) as avg_spent
FROM customers c
JOIN orders o
ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.customer_name
ORDER BY c.customer_name